# Paths nested inside containers: the usual UX, and the edges

[Files.ipynb](Files.ipynb) shows the sunny-day behaviour of `Path` arguments and
return values: files and directories are stored by **content** and rematerialized
under a temporary path on a cache hit.

This notebook explores what happens when paths are *nested inside other values* —
dicts, lists, dataclasses — which fleche's `DestructuringMixin` takes apart and
reassembles ("mends") around the path machinery.  The first half is the intended
UX; the second half collects the edge cases where a cache hit is **not** a faithful
replay of the original call: location changes, aliasing loss, path dict-keys, and
container types whose mending is incomplete or outright broken.

In [1]:
import gc
import tempfile
from collections import Counter, defaultdict, namedtuple
from dataclasses import dataclass
from pathlib import Path

import fleche as fl
from fleche import fleche

fl.cache("memory")   # transient in-memory cache
c = fl.cache()

# DestructuringMixin sits *above* PathValueMixin: containers are taken apart
# first, and each nested Path is then stored by content as a blob.
[k.__name__ for k in type(c.values).__mro__ if k.__name__.endswith("Mixin")]

['PerKeyLockMixin', 'DestructuringMixin', 'PathValueMixin', 'ValueMixin']

In [2]:
# A scratch directory standing in for "wherever your functions write their files".
WORK = Path(tempfile.mkdtemp(suffix="-fleche-nested"))

def fresh(name, text):
    p = WORK / name
    p.write_text(text)
    return p

## The usual UX: containers of paths just work

A cached function can return paths tucked inside dicts, lists, tuples, or
dataclasses.  On a hit, the container is mended and every nested path comes back
freshly materialized, with its content and basename intact.

In [3]:
@dataclass
class FitResult:
    report: Path
    score: float

@fleche
def fit(seed):
    print("  [fit] running:", seed)
    return {
        "results": [FitResult(fresh(f"{seed}-fit.txt", f"loss={len(seed)}"), 0.5)],
        "seed": seed,
    }

cold = fit("alpha")
print("cold:", type(cold["results"][0].report).__name__, "->", cold["results"][0].report)

  [fit] running: alpha
cold: PosixPath -> /tmp/claude-1000/tmpddupkllp-fleche-nested/alpha-fit.txt


In [4]:
warm = fit("alpha")   # no "[fit] running" -> served from cache
r = warm["results"][0]
print("warm:", type(r.report).__name__, "->", r.report)
print("content:", r.report.read_text(), "| name kept:", r.report.name)

warm: TempPath -> /tmp/claude-1000/tmphvfdmrplfleche/alpha-fit.txt
content: loss=5 | name kept: alpha-fit.txt


True content addressing: the *original* file can vanish entirely — the cache holds
the bytes, so hits keep working.

In [5]:
cold["results"][0].report.unlink()          # delete the original on disk
again = fit("alpha")
again["results"][0].report.read_text()      # still served, from stored content

'loss=5'

## Edge 1: a hit changes *where* (and what type) the path is

The cold call returns whatever the function returned — the real location in
`WORK`, as a plain `Path`.  A warm hit returns a `TempPath` in a fresh temporary
directory.  Content is identical; **location is not**.

The classic footgun: code that resolves *siblings* of a returned path
(`p.parent / "meta.json"`) works on the first call and breaks on every hit,
because the materialized file sits alone in its temp directory.

In [6]:
@fleche
def produce(seed):
    print("  [produce] running:", seed)
    fresh(f"{seed}-meta.json", '{"version": 1}')   # sibling, NOT returned
    return fresh(f"{seed}-data.csv", "1,2,3")

p_cold = produce("beta")
print("cold sibling exists:", (p_cold.parent / "beta-meta.json").exists())

p_warm = produce("beta")
print("warm location:", p_warm)
print("warm sibling exists:", (p_warm.parent / "beta-meta.json").exists())

  [produce] running: beta
cold sibling exists: True
warm location: /tmp/claude-1000/tmptm88ttu4fleche/beta-data.csv
warm sibling exists: False


Only what is *returned* (or passed) is captured.  If the sibling matters, return
it too — or return the whole directory.

## Edge 2: aliasing is not preserved

Return the *same* path twice and the cold result holds one object in two slots.
The warm hit mends each slot independently: two separate materializations, in two
different temp directories.  Equal content, unequal (and non-identical) paths.

In [7]:
@fleche
def twice(seed):
    print("  [twice] running:", seed)
    p = fresh(f"{seed}-shared.txt", seed * 2)
    return [p, p]

pc = twice("gamma")
print("cold:  identical:", pc[0] is pc[1], "| equal:", pc[0] == pc[1])

pw = twice("gamma")
print("warm:  identical:", pw[0] is pw[1], "| equal:", pw[0] == pw[1],
      "| same content:", pw[0].read_bytes() == pw[1].read_bytes())

  [twice] running: gamma
cold:  identical: True | equal: True
warm:  identical: False | equal: False | same content: True


## Edge 3: paths as dict *keys* mend into different keys

Dict keys are destructured like values.  A `Path` key comes back as a `TempPath`
at a new location — so the mended dict has a *different key* than the original,
and lookups by the original path miss.  Use `str(path)` or a stable identifier as
the key if you intend to look things up by it.

In [8]:
@fleche
def index(seed):
    print("  [index] running:", seed)
    return {fresh(f"{seed}-k.txt", seed): "metadata"}

kc = index("delta")
orig_key = next(iter(kc))

kw = index("delta")
warm_key = next(iter(kw))
print("cold key:", orig_key)
print("warm key:", warm_key)
print("kw[orig_key] works:", orig_key in kw)

  [index] running: delta
cold key: /tmp/claude-1000/tmpddupkllp-fleche-nested/delta-k.txt
warm key: /tmp/claude-1000/tmpxk4r0zg6fleche/delta-k.txt
kw[orig_key] works: False


## Edge 4: every hit materializes a fresh copy, with temp-file lifetime

Each hit copies the stored bytes into a new temporary directory (large files: mind
the churn).  The temp tree lives exactly as long as some `TempPath` derived from
it is referenced — keep only a `str` of the location and the file is gone once the
path object is collected.

In [9]:
@fleche
def artifact(seed):
    print("  [artifact] running:", seed)
    return fresh(f"{seed}-art.txt", seed)

_ = artifact("epsilon")            # cold
w1 = artifact("epsilon")           # hit -> copy #1
w2 = artifact("epsilon")           # hit -> copy #2
print("two hits, same location:", w1 == w2)

location = str(w1)                 # keep only the string...
del w1
gc.collect()
print("file still there after dropping the TempPath:", Path(location).exists())
print("w2 unaffected (own temp dir):", w2.exists())

  [artifact] running: epsilon
two hits, same location: False
file still there after dropping the TempPath: False
w2 unaffected (own temp dir): True


## Edge 5: paths hidden in *opaque* containers are stored by location, not content

Destructuring only recurses into what it knows: lists, tuples, dicts, dataclasses,
attrs classes.  Everything else — namedtuples (deliberately treated as opaque),
sets, arbitrary objects with a `Path` attribute — is stored **verbatim**.  The
nested path never reaches the content machinery: what is stored is the path
*object*, pointing at the original location.

The call is still *keyed* correctly (the digest layer does recurse, hashing file
content), so hits and misses behave right.  But a warm hit hands back the original
location — an **incompletely mended** result.  If that file has been deleted,
moved, or edited since, the hit returns a dangling or stale path, silently.

In [10]:
Bundle = namedtuple("Bundle", ["out", "score"])

@fleche
def bundle(seed):
    print("  [bundle] running:", seed)
    return Bundle(fresh(f"{seed}-nt.txt", seed), 0.5)

b_cold = bundle("zeta")
b_warm = bundle("zeta")   # cache hit...
print("warm type:", type(b_warm.out).__name__, "->", b_warm.out)
print("points at the ORIGINAL location:", b_warm.out == b_cold.out)

  [bundle] running: zeta
warm type: PosixPath -> /tmp/claude-1000/tmpddupkllp-fleche-nested/zeta-nt.txt
points at the ORIGINAL location: True


In [11]:
# Now the original vanishes -- e.g. a scratch dir is cleaned up between sessions.
b_cold.out.unlink()

b_stale = bundle("zeta")           # still a cache hit (keyed on content at save time)
print("hit returns:", b_stale.out)
print("exists:", b_stale.out.exists(), " <- dangling, no warning")

hit returns: /tmp/claude-1000/tmpddupkllp-fleche-nested/zeta-nt.txt
exists: False  <- dangling, no warning


Compare with Edge 1's dict: the *destructured* container survived deletion of the
original because the content was captured.  The namedtuple did not.  Same story
for sets and custom non-dataclass objects.

**Rule of thumb:** return paths in plain dicts/lists/tuples/dataclasses, not
smuggled inside opaque types.

## Edge 6: container subclasses are opaque — deliberately

Mending rebuilds containers via `type(value)(<children>)`, a contract subclasses
may repurpose: `defaultdict`'s first argument is a factory (would crash),
`Counter` *counts* its argument (would silently corrupt).  Destructuring therefore
matches **exact types only** — `dict`, `OrderedDict`, `list`, `tuple` (plus
dataclasses/attrs, whose mending bypasses `__init__`).  Everything else is stored
verbatim as an opaque value.

So subclasses round-trip *as values* — but any path nested inside them follows
Edge 5's location semantics, not content addressing.

In [12]:
@fleche
def by_kind(seed):
    print("  [by_kind] running:", seed)
    d = defaultdict(list)
    d["files"].append(fresh(f"{seed}-dd.txt", seed))
    return d

dd_cold = by_kind("eta")
dd_warm = by_kind("eta")
print("warm:", type(dd_warm).__name__, dict(dd_warm))
print("but the nested path is the ORIGINAL location:", dd_warm["files"][0] == dd_cold["files"][0])

  [by_kind] running: eta
warm: defaultdict {'files': [PosixPath('/tmp/claude-1000/tmpddupkllp-fleche-nested/eta-dd.txt')]}
but the nested path is the ORIGINAL location: True


In [13]:
@fleche
def tally(seed):
    print("  [tally] running:", seed)
    return Counter({"a": seed, "b": seed + 1})

print("cold:", tally(5))
print("warm:", tally(5), " <- opaque, so counts survive intact")

# Custom destructurers for subclasses can be opted in later:
help(fl.storage.destructuring.register_destructurer)

  [tally] running: 5
cold: Counter({'b': 6, 'a': 5})
warm: Counter({'b': 6, 'a': 5})  <- opaque, so counts survive intact
Help on function register_destructurer in module fleche.storage.destructuring:

register_destructurer(pred: Callable[[Any], bool], fn: Callable) -> None
    Register a custom container destructurer.

    *pred(value)* should return ``True`` for values this destructurer handles.
    *fn* must accept ``(intern, value)`` where *intern* is
    :meth:`DestructuringMixin._intern_rec`.  Entries are appended after the
    built-in ones; first match wins, so registering a handler for an entirely
    new container type is safe without displacing list/dict/dataclass/attrs.
    The built-in predicates match exact types only, so a handler for a subclass
    (e.g. ``defaultdict`` with a picklable factory) can also be registered
    without conflict.  Call before any :class:`DestructuringMixin` instance is
    used.



## Summary: rules of thumb

- **Works out of the box:** paths (files *and* directories) as arguments, return
  values, or nested anywhere inside dicts / lists / tuples / dataclasses / attrs
  classes — arbitrarily deep.  Content-addressed, dedup'd, survives deletion of
  the originals.
- **A hit is a copy, not a replay:** returned paths live in fresh temp
  directories.  Don't resolve siblings, don't compare locations, don't expect
  aliasing, and keep a reference to the `Path` object for as long as you need the
  file.
- **Don't key dicts by `Path`** if you'll look them up afterwards — keys mend
  into new locations.  Use `str(path)` or a stable ID.
- **Don't hide paths in opaque containers** (namedtuples, sets, plain classes,
  and any container *subclass* — only exact `dict` / `OrderedDict` / `list` /
  `tuple` are destructured): they are stored by location and come back stale or
  dangling after the original moves on.  `register_destructurer` is the opt-in
  door for well-behaved custom containers.